# Population Health Analytics Dashboard 
Python • Pandas • Plotly • Synthetic EHR (Synthea) • Executive Analytics

This notebook demonstrates an end-to-end healthcare analytics workflow, from raw EHR ingestion through executive reporting and consultant interpretation.

**Purpose:** Provide an executive-level overview of the synthetic healthcare population, utilization, clinical burden, and payment activity.

**Data source:** Synthea-generated synthetic claims and clinical data  

**Population:** Synthetic patients; no protected health information  

**Question it answers:** What does this healthcare population look like?

# Executive Summary
| Executive KPI        | Value                                          |
| -------------------- | ---------------------------------------------- |
| Population           | 16 patients                                    |
| Study period         | 2020–2025                                      |
| Encounters           | 1,494                                          |
| Total charges        | $2.95M                                         |
| Top diagnosis        | Hypertension                                   |
| Highest-cost patient | $654,933                                       |
| Assessment           | Payment activity appears internally consistent |


Notebook Workflow

✓ Load synthetic EHR data

✓ Perform data quality checks

✓ Calculate executive KPIs

✓ Analyze population characteristics

✓ Summarize healthcare utilization

✓ Evaluate financial performance

✓ Generate consultant interpretation

# Data Loading & Preparation

In [13]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display

PROJECT_ROOT = Path(r"C:\AI Projects\PopulationHealthWorkbench")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_tables

tables = load_tables()

print(f"Loaded {len(tables)} tables.")

Loaded 18 tables.


# Assign the principal tables

In [14]:
patients = tables["patients"].copy()
encounters = tables["encounters"].copy()
conditions = tables["conditions"].copy()
claims = tables["claims"].copy()
claims_transactions = tables["claims_transactions"].copy()
procedures = tables["procedures"].copy()
medications = tables["medications"].copy()
observations = tables["observations"].copy()
providers = tables["providers"].copy()
organizations = tables["organizations"].copy()
payers = tables["payers"].copy()

# Prepare dates and numeric fields

In [18]:
patients["BIRTHDATE"] = pd.to_datetime(
    patients["BIRTHDATE"],
    errors="coerce"
)

encounters["START"] = pd.to_datetime(
    encounters["START"],
    errors="coerce",
    utc=True
)

claims_transactions["AMOUNT"] = pd.to_numeric(
    claims_transactions["AMOUNT"],
    errors="coerce"
)

claims_transactions["PAYMENTS"] = pd.to_numeric(
    claims_transactions["PAYMENTS"],
    errors="coerce"
)

analysis_date = encounters["START"].max().tz_localize(None)

age_years = (
    analysis_date - patients["BIRTHDATE"]
).dt.days / 365.25

patients["AGE"] = np.floor(age_years).astype("Int64")

print("Analysis date:", analysis_date.date())

Analysis date: 2026-07-24


# Calculate core KPIs


In [19]:
charge_rows = claims_transactions[
    claims_transactions["TYPE"].eq("CHARGE")
].copy()

payment_rows = claims_transactions[
    claims_transactions["TYPE"].eq("PAYMENT")
].copy()

total_patients = patients["Id"].nunique()
total_encounters = encounters["Id"].nunique()
total_claims = claims["Id"].nunique()
total_charges = charge_rows["AMOUNT"].sum()
total_payments = payment_rows["PAYMENTS"].sum()

payment_ratio = (
    total_payments / total_charges
    if total_charges != 0
    else np.nan
)

average_cost_per_patient = (
    total_charges / total_patients
    if total_patients != 0
    else np.nan
)

average_encounters_per_patient = (
    total_encounters / total_patients
    if total_patients != 0
    else np.nan
)

# Display KPI cards

In [20]:
def format_currency(value):
    return f"${value:,.0f}"

def format_number(value):
    return f"{value:,.0f}"

def format_decimal(value):
    return f"{value:,.1f}"

kpi_cards = [
    ("Patients", format_number(total_patients)),
    ("Encounters", format_number(total_encounters)),
    ("Claims", format_number(total_claims)),
    ("Total Charges", format_currency(total_charges)),
    ("Total Payments", format_currency(total_payments)),
    ("Payment-to-Charge", f"{payment_ratio:.1%}"),
    ("Average Cost per Patient", format_currency(average_cost_per_patient)),
    ("Encounters per Patient", format_decimal(average_encounters_per_patient)),
]

cards_html = """
<div style="
    display:grid;
    grid-template-columns:repeat(4, minmax(180px, 1fr));
    gap:14px;
    margin:10px 0 25px 0;
">
"""

for label, value in kpi_cards:
    cards_html += f"""
    <div style="
        border:1px solid #d8dde3;
        border-radius:10px;
        padding:16px;
        background:#ffffff;
        box-shadow:0 2px 6px rgba(0,0,0,0.06);
    ">
        <div style="
            font-size:13px;
            color:#5f6b76;
            margin-bottom:8px;
        ">{label}</div>
        <div style="
            font-size:24px;
            font-weight:700;
            color:#1f2933;
        ">{value}</div>
    </div>
    """

cards_html += "</div>"

display(HTML(cards_html))

# Population profile

In [21]:
population_summary = pd.DataFrame({
    "Metric": [
        "Average age",
        "Median age",
        "Youngest patient",
        "Oldest patient"
    ],
    "Value": [
        round(patients["AGE"].mean(), 1),
        round(patients["AGE"].median(), 1),
        patients["AGE"].min(),
        patients["AGE"].max()
    ]
})

population_summary

,Metric,Value
0,Average age,61.6
1,Median age,76.5
2,Youngest patient,5.0
3,Oldest patient,77.0


## Age distribution

In [22]:
fig = px.histogram(
    patients,
    x="AGE",
    nbins=10,
    title="Patient Age Distribution",
    labels={"AGE": "Age"}
)

fig.update_layout(
    yaxis_title="Number of Patients",
    template="plotly_white"
)

fig.show()

## Gender distribution

In [23]:
gender_distribution = (
    patients["GENDER"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("Gender")
    .reset_index(name="Patients")
)

fig = px.pie(
    gender_distribution,
    names="Gender",
    values="Patients",
    title="Patient Distribution by Gender",
    hole=0.4
)

fig.update_layout(template="plotly_white")
fig.show()

## Race distribution

In [24]:
race_distribution = (
    patients["RACE"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("Race")
    .reset_index(name="Patients")
)

fig = px.bar(
    race_distribution,
    x="Race",
    y="Patients",
    title="Patient Distribution by Race",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Race",
    yaxis_title="Patients",
    template="plotly_white"
)

fig.show()

## Utilization profile

In [25]:
encounter_mix = (
    encounters["ENCOUNTERCLASS"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("Encounter Class")
    .reset_index(name="Encounters")
)

fig = px.bar(
    encounter_mix,
    x="Encounter Class",
    y="Encounters",
    title="Encounter Mix",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Encounter Class",
    yaxis_title="Number of Encounters",
    template="plotly_white"
)

fig.show()

## Top conditions

In [26]:
top_conditions = (
    conditions
    .groupby("DESCRIPTION")
    .agg(
        Condition_Records=("DESCRIPTION", "size"),
        Unique_Patients=("PATIENT", "nunique")
    )
    .reset_index()
    .sort_values(
        ["Unique_Patients", "Condition_Records"],
        ascending=False
    )
    .head(10)
)

top_conditions

,DESCRIPTION,Condition_Records,Unique_Patients
60,Medication review due (situation),109,16
95,Stress (finding),56,15
37,Full-time employment (finding),55,15
39,Gingivitis (disorder),53,14
90,Social isolation (finding),31,14
73,Part-time employment (finding),33,12
101,Viral sinusitis (disorder),17,11
12,Body mass index 30+ - obesity (finding),11,11
31,Educated to high school level (finding),11,11
100,Victim of intimate partner abuse (finding),18,10


In [27]:
fig = px.bar(
    top_conditions.sort_values("Unique_Patients"),
    x="Unique_Patients",
    y="DESCRIPTION",
    orientation="h",
    title="Top Conditions by Number of Patients",
    text_auto=True,
    labels={
        "DESCRIPTION": "Condition",
        "Unique_Patients": "Unique Patients"
    }
)

fig.update_layout(template="plotly_white")
fig.show()

## Top procedures

In [28]:
top_procedures = (
    procedures["DESCRIPTION"]
    .value_counts()
    .head(10)
    .rename_axis("Procedure")
    .reset_index(name="Count")
)

fig = px.bar(
    top_procedures.sort_values("Count"),
    x="Count",
    y="Procedure",
    orientation="h",
    title="Top 10 Procedures",
    text_auto=True
)

fig.update_layout(template="plotly_white")
fig.show()

## Charges and payments

In [36]:
financial_summary = pd.DataFrame({
    "Financial Measure": [
        "Charges",
        "Payments"
    ],
    "Amount": [
        total_charges,
        total_payments
    ]
})

fig = px.bar(
    financial_summary,
    x="Financial Measure",
    y="Amount",
    title="Charges Compared with Payments"
)

fig.update_traces(
    marker_color=["steelblue", "seagreen"],
    text=[f"${x:,.0f}" for x in financial_summary["Amount"]],
    textposition="outside",
    cliponaxis=False
)

fig.update_layout(
    yaxis_title="Amount (USD)",
    xaxis_title="Financial Measure",
    template="plotly_white",
    yaxis_tickprefix="$",
    yaxis_tickformat=",.0f",
    yaxis_range=[
        0,
        financial_summary["Amount"].max() * 1.15
    ],
    margin=dict(t=100, r=40, b=60, l=90)
)

fig.show()

## High-cost patients

In [30]:
patient_costs = (
    charge_rows
    .groupby("PATIENTID", as_index=False)["AMOUNT"]
    .sum()
    .rename(columns={"AMOUNT": "Total_Charges"})
    .sort_values("Total_Charges", ascending=False)
)

patient_names = patients[
    ["Id", "FIRST", "LAST"]
].copy()

patient_names["Patient"] = (
    patient_names["FIRST"].fillna("") +
    " " +
    patient_names["LAST"].fillna("")
).str.strip()

patient_costs = patient_costs.merge(
    patient_names[["Id", "Patient"]],
    left_on="PATIENTID",
    right_on="Id",
    how="left"
)

top_cost_patients = patient_costs.head(10)

top_cost_patients[
    ["Patient", "Total_Charges"]
].style.format({
    "Total_Charges": "${:,.2f}"
})

,Patient,Total_Charges
0,Alonso270 Doyle959,"$654,932.51"
1,Fausto876 Nader710,"$446,026.94"
2,Dennis979 Zulauf375,"$335,889.95"
3,Tillie335 Dach178,"$235,632.59"
4,Alicia629 Fisher429,"$195,565.68"
5,Minh326 Barton704,"$193,037.28"
6,Rueben647 Aufderhar910,"$192,642.41"
7,Monty345 Price929,"$129,114.71"
8,John539 Ferry570,"$122,616.88"
9,Elbert916 Koepp521,"$102,206.34"


## Automated findings

In [37]:
top_condition = (
    top_conditions.iloc[0]["DESCRIPTION"]
    if not top_conditions.empty
    else "Not available"
)

top_encounter_class = (
    encounter_mix.iloc[0]["Encounter Class"]
    if not encounter_mix.empty
    else "Not available"
)

highest_cost_patient_amount = (
    patient_costs.iloc[0]["Total_Charges"]
    if not patient_costs.empty
    else np.nan
)

findings_html = f"""
<div style="
    border-left:5px solid #4b6f8a;
    background:#f4f7f9;
    padding:18px 22px;
    border-radius:6px;
    margin-top:20px;
">
    <h3 style="margin-top:0;">Executive Findings</h3>
    <ul>
        <li>The dataset represents <strong>{total_patients:,}</strong>
            synthetic patients and <strong>{total_encounters:,}</strong>
            encounters.</li>
        <li>Total billed charges were
            <strong>{format_currency(total_charges)}</strong>, with recorded
            payments of <strong>{format_currency(total_payments)}</strong>.</li>
        <li>The payment-to-charge ratio was
            <strong>{payment_ratio:.1%}</strong>.</li>
        <li>The most prevalent condition by unique patient count was
            <strong>{top_condition}</strong>.</li>
        <li>The most common encounter setting was
            <strong>{top_encounter_class}</strong>.</li>
        <li>The highest-cost synthetic patient accumulated
            <strong>{format_currency(highest_cost_patient_amount)}</strong>
            in charges.</li>
    </ul>
</div>
"""

display(HTML(findings_html))

# Dynamic Consultant Interpretation


In [ ]:
from IPython.display import Markdown, display

consultant_summary = f"""
# Consultant Interpretation

## Key observations

- This synthetic healthcare population contains **{total_patients:,} patients** with **{total_encounters:,} healthcare encounters**, representing a diverse mix of outpatient, emergency, inpatient, and preventive services.

- The most frequently documented clinical condition was **{top_condition}**, suggesting it represents the largest disease cohort in this demonstration dataset.

- Total healthcare charges reached **{format_currency(total_charges)}**, while recorded payments totaled **{format_currency(total_payments)}**, resulting in a payment-to-charge ratio of **{payment_ratio:.1%}**.

- The highest-cost patient accumulated **{format_currency(highest_cost_patient_amount)}** in total charges, indicating that healthcare spending is concentrated among a relatively small number of patients, a common pattern observed in population health analytics.

- **{top_encounter_class}** was the predominant encounter setting, indicating where most healthcare utilization occurred within this synthetic population.

## Consultant Assessment

Overall, the financial data appear internally consistent, with payment activity generally aligned with billed charges for this demonstration dataset. No obvious large-scale payment imbalance is evident from the aggregate summary, although patient-level and claim-level analyses are recommended to identify potential unpaid claims, payment delays, or unusual billing patterns.

The dataset provides a realistic foundation for demonstrating healthcare analytics workflows including population health management, utilization analysis, claims analytics, provider benchmarking, payment integrity, and AI-assisted decision support.

## Recommended Follow-up Analyses

1. Risk-adjust patient costs using chronic disease burden.
2. Identify high-utilization patients suitable for care management.
3. Compare organizations and providers on cost, utilization, and outcomes.
4. Detect payment anomalies, unpaid claims, and duplicate transactions.
5. Develop chronic disease cohorts for targeted population health interventions.
6. Build predictive models to identify future high-cost patients.
7. Develop executive dashboards using Power BI or Streamlit for operational monitoring.

## Important Limitation

This analysis uses **synthetic Synthea-generated data** intended for education and software development. The findings demonstrate analytical methodology and should **not** be interpreted as evidence regarding any real healthcare population.
"""

display(Markdown(consultant_summary))


# Consultant Interpretation

## Key observations

- This synthetic healthcare population contains **16 patients** with **1,494 healthcare encounters**, representing a diverse mix of outpatient, emergency, inpatient, and preventive services.

- The most frequently documented clinical condition was **Medication review due (situation)**, suggesting it represents the largest disease cohort in this demonstration dataset.

- Total healthcare charges reached **$2,946,806**, while recorded payments totaled **$2,946,806**, resulting in a payment-to-charge ratio of **100.0%**.

- The highest-cost patient accumulated **$654,933** in total charges, indicating that healthcare spending is concentrated among a relatively small number of patients, a common pattern observed in population health analytics.

- **ambulatory** was the predominant encounter setting, indicating where most healthcare utilization occurred within this synthetic population.

## Consultant Assessment

Overall, the financial data appear internally consistent, with payment activity generally aligned with billed charges for this demonstration dataset. No obvious large-scale payment imbalance is evident from the aggregate summary, although patient-level and claim-level analyses are recommended to identify potential unpaid claims, payment delays, or unusual billing patterns.

The dataset provides a realistic foundation for demonstrating healthcare analytics workflows including population health management, utilization analysis, claims analytics, provider benchmarking, payment integrity, and AI-assisted decision support.

## Recommended Follow-up Analyses

1. Risk-adjust patient costs using chronic disease burden.
2. Identify high-utilization patients suitable for care management.
3. Compare organizations and providers on cost, utilization, and outcomes.
4. Detect payment anomalies, unpaid claims, and duplicate transactions.
5. Develop chronic disease cohorts for targeted population health interventions.
6. Build predictive models to identify future high-cost patients.
7. Develop executive dashboards using Power BI or Streamlit for operational monitoring.

## Important Limitation

This analysis uses **synthetic Synthea-generated data** intended for education and software development. The findings demonstrate analytical methodology and should **not** be interpreted as evidence regarding any real healthcare population.
